# 06 — Serie temporali I: previsioni di riferimento, autocorrelazione, AR (1927–1938)
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

Percorso in quattro notebook che segue la cronologia di `refs/timeseries/time-series-forecasting-01.jpg` — *una cronologia di idee, non una classifica di modelli*:

| Anni | Idea | Notebook | API `MathemaSharp.TimeSeries` |
|---|---|---|---|
| — | Mean, naive, seasonal naive | 06 | `ReferenceForecasts`, `Accuracy` |
| 1927–1938 | Yule (AR), Wold | 06 | `SeriesStatistics`, `ArModel` |
| 1950s–1960 | SES, Holt, Holt-Winters | 07 | `ExponentialSmoothing` |
| 1960 | Kalman | 07 | `KalmanFilter` |
| 1985 | Damped trend | 07 | `ExponentialSmoothing.Holt(..., phi)` |
| 1970 | Box-Jenkins / ARIMA | 08 | `ArimaModel` |
| 1980 | VAR | 08 | `VarModel` |
| 1982/1986 | ARCH/GARCH | 08 | `Garch11` |
| 1990 | STL | 09 | `Stl` |

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// Serie mensile sintetica: livello, trend lineare, stagionalità annuale, rumore gaussiano. Seme fisso: risultati riproducibili.
var random = new Random(2026);
const int period = 12;
var y = Enumerable.Range(0, 120).Select(t => 100 + 0.6 * t + 8 * Math.Sin(2 * Math.PI * t / period) + 3 * Simulation.StandardNormal(random)).ToArray();
double[] train = y[..108], test = y[108..];
Chart.Lines("Serie mensile sintetica (120 mesi)", new Line("y", y));

## Previsioni di riferimento
Sono ancora i benchmark di oggi: un modello che non le batte non giustifica la sua complessità.

- **Mean**: $\hat y_{T+h} = \bar y$ — ottima se la serie è rumore bianco attorno a una media.
- **Naive**: $\hat y_{T+h} = y_T$ — ottima per una passeggiata aleatoria.
- **Seasonal naive**: $\hat y_{T+h} = y_{T+h-m(k+1)}$, $k=\lfloor (h-1)/m \rfloor$.
- **Drift**: $\hat y_{T+h} = y_T + h\,\frac{y_T-y_1}{T-1}$.

Per confrontarle si usa la **MASE** (Hyndman & Koehler 2006): MAE fuori campione diviso il MAE in-sample del seasonal naive. È indipendente dalla scala, definita con gli zeri; MASE < 1 significa battere il naive stagionale *in-sample*.

In [ ]:
var forecasts = new (string Name, double[] F)[]
{
    ("Mean", ReferenceForecasts.Mean(train, 12)),
    ("Naive", ReferenceForecasts.Naive(train, 12)),
    ("Seasonal naive", ReferenceForecasts.SeasonalNaive(train, period, 12)),
    ("Drift", ReferenceForecasts.Drift(train, 12)),
};
Table.Show(new[] { "Metodo", "MAE", "RMSE", "MAPE %", "MASE (m=12)" },
    forecasts.Select(f => new object[] { f.Name, Accuracy.Mae(test, f.F), Accuracy.Rmse(test, f.F), Accuracy.Mape(test, f.F), Accuracy.Mase(test, f.F, train, period) }));
Chart.Lines("Ultimi 3 anni + previsioni a 12 mesi",
    new Line("osservato", y[84..]), new Line("seasonal naive", forecasts[2].F, Start: 24, Dashed: true), new Line("drift", forecasts[3].F, Start: 24, Dashed: true));

## Autocovarianza, ACF e PACF
$$\hat\gamma(k) = \frac1n \sum_{t=1}^{n-k} (y_t-\bar y)(y_{t+k}-\bar y), \qquad \hat\rho(k)=\hat\gamma(k)/\hat\gamma(0).$$
Il divisore è $n$, non $n-k$: lo stimatore è distorto, ma la matrice di Toeplitz $\Gamma_{ij}=\hat\gamma(|i-j|)$ resta semidefinita positiva. È questa proprietà che garantisce $|\text{PACF}|\le 1$ e un AR stimato stazionario.

La **PACF** al lag $k$ è l'ultimo coefficiente $\phi_{kk}$ dell'AR($k$) che meglio approssima la serie. Si ottiene dalla ricorsione di **Levinson-Durbin**, che risolve le equazioni di Yule-Walker $\Gamma\phi=\gamma$ in $O(p^2)$:
$$\kappa_k = \frac{\gamma(k) - \sum_{j<k}\phi_{k-1,j}\,\gamma(k-j)}{v_{k-1}},\quad \phi_{k,j}=\phi_{k-1,j}-\kappa_k\phi_{k-1,k-j},\quad v_k=v_{k-1}(1-\kappa_k^2).$$
La banda $\pm 1.96/\sqrt n$ è il riferimento del rumore bianco. Firma tipica: un AR($p$) ha ACF che decade e PACF che si **taglia** dopo il lag $p$.

In [ ]:
// Yule (1927) modellò le macchie solari con un AR(2). φ = (1.6, −0.9) ha radici complesse: pseudo-ciclo.
var ar2 = Simulation.Arma<double>(new Random(1927), 600, 0, new[] { 1.6, -0.9 }, Array.Empty<double>(), 1);
var band = 1.96 / Math.Sqrt(ar2.Length);
Chart.Lines("AR(2) simulato, φ = (1.6, −0.9)", new Line("y", ar2[..200]));
Chart.Stems("ACF: oscillazione smorzata, periodo ≈ 11", SeriesStatistics.Acf(ar2, 30)[1..], band);
Chart.Stems("PACF: si taglia dopo il lag 2", SeriesStatistics.Pacf(ar2, 30), band);

## Yule-Walker e il ciclo nascosto nelle radici
Per un AR(2) $y_t = \phi_1 y_{t-1} + \phi_2 y_{t-2} + \varepsilon_t$ il polinomio caratteristico $z^2-\phi_1 z-\phi_2$ ha radici complesse quando $\phi_1^2+4\phi_2<0$. Le radici sono $r e^{\pm i\theta}$ con $r=\sqrt{-\phi_2}$ e $\cos\theta = \phi_1/(2\sqrt{-\phi_2})$: l'ACF è una sinusoide smorzata di periodo $2\pi/\theta$.

Con $\phi=(1.6,-0.9)$: $\cos\theta = 1.6/(2\cdot 0.949) = 0.843$, quindi $\theta \approx 0.567$ e un periodo di circa **11.1** passi. È l'idea di Yule: un ciclo di circa 11 anni senza alcuna sinusoide deterministica, solo dinamica autoregressiva guidata da shock.

**Wold (1938)**: ogni processo stazionario a media nulla si scrive come $\sum_j \psi_j \varepsilon_{t-j}$ più una parte deterministica. È un teorema, non un modello: l'AR finito ne è un'approssimazione parsimoniosa.

In [ ]:
var yw = ArModel.FitYuleWalker(ar2, 2);
double Period(double phi1, double phi2) => 2 * Math.PI / Math.Acos(phi1 / (2 * Math.Sqrt(-phi2)));
Table.Show(new[] { "", "φ1", "φ2", "σ²", "periodo implicito" }, new[]
{
    new object[] { "vero", 1.6, -0.9, 1.0, Period(1.6, -0.9) },
    new object[] { "Yule-Walker", yw.Phi[0], yw.Phi[1], yw.InnovationVariance, Period(yw.Phi[0], yw.Phi[1]) },
});
Chart.Lines("Previsione AR(2): il ciclo si smorza verso la media", new Line("osservato", ar2[^60..]), new Line("previsione", yw.Forecast(ar2, 40), Start: 60, Dashed: true));

## Controllo dei residui: Ljung-Box
$Q = n(n+2)\sum_{k=1}^{h}\hat\rho_k^2/(n-k) \sim \chi^2_{h-p}$ se i residui sono rumore bianco. Un p-value piccolo indica che il modello non ha catturato tutta la dinamica. Il p-value usa la gamma incompleta regolarizzata, implementata in `SpecialFunctions` (Lanczos + serie/frazione continua).

In [ ]:
Table.Show(new[] { "Modello", "σ² residua", "Ljung-Box Q(20)", "p-value" }, Enumerable.Range(1, 3).Select(p =>
{
    var model = ArimaModel.FitHannanRissanen(ar2, p, 0, 0);
    var residuals = model.Residuals(ar2)[p..];
    var lb = SeriesStatistics.LjungBox(residuals, 20, p);
    return new object[] { $"AR({p})", model.InnovationVariance, lb.Statistic, lb.PValue };
}));

## Esperimenti
- Portare φ2 a −0.5: le radici diventano più interne al cerchio unitario e il ciclo si smorza prima.
- φ = (0.5, 0.45): radici reali, nessun ciclo; quale firma hanno ACF e PACF?
- Applicare la seasonal naive a una serie senza stagionalità: la MASE resta sotto 1?